# Lab 10 solution, notebook 2: reject a refund with a reason, and let small refunds through

The worked answers to the two challenge tasks of `10b_langgraph_at_the_desk.ipynb`. The setup repeats the
notebook's lookups, tools, nodes and Part B's graph, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import sqlite3

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def find_customer(name: str) -> list:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

The refund table of this file is `outputs/refund_requests.db` inside `solved/`, emptied on every run.

In [3]:
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

refunds = sqlite3.connect(OUTPUTS / "refund_requests.db")
refunds.execute("CREATE TABLE IF NOT EXISTS refund_requests "
                "(request_id INTEGER PRIMARY KEY, invoice_id INTEGER, amount REAL, reason TEXT)")
refunds.execute("DELETE FROM refund_requests")
refunds.commit()


def file_refund_request(invoice_id: int, amount: float, reason: str) -> str:
    """File a refund request for one invoice, in US dollars. A person reviews it before any money moves."""
    cursor = refunds.execute("INSERT INTO refund_requests (invoice_id, amount, reason) VALUES (?, ?, ?)",
                             (invoice_id, amount, reason))
    refunds.commit()
    return f"refund request {cursor.lastrowid} filed: invoice {invoice_id}, {amount} US dollars"


def show_requests():
    rows = refunds.execute("SELECT * FROM refund_requests").fetchall()
    print("refund_requests:", rows or "empty")

In [4]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

SYSTEM = ("You help the support reps of a music shop. Look up every fact with the tools; never answer from memory. "
          "Answer in one or two sentences.")

llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT, timeout=60, max_retries=2)

TOOLS = [tool(find_customer), tool(get_invoices), tool(get_support_rep), tool(file_refund_request)]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
model_with_tools = llm.bind_tools(TOOLS)


def show(messages):
    """Print a conversation: one line per message, with who it came from."""
    for message in messages:
        if message.type == "ai" and message.tool_calls:
            print("ai     asks for", [(call["name"], call["args"]) for call in message.tool_calls])
        elif message.type == "ai":
            print("ai    ", message.text)
        else:
            print(f"{message.type:6}", message.content)

In [5]:
class DeskState(TypedDict):
    messages: Annotated[list, add_messages]


def model_node(state):
    reply = model_with_tools.invoke([SystemMessage(SYSTEM)] + state["messages"])
    return {"messages": [reply]}


def tools_node(state):
    results = []
    for call in state["messages"][-1].tool_calls:
        results.append(TOOLS_BY_NAME[call["name"]].invoke(call))
    return {"messages": results}


def review(state):
    calls = state["messages"][-1].tool_calls
    decision = interrupt([{"tool": call["name"], "args": call["args"]} for call in calls])

    if decision == "approve":
        return Command(goto="tools")

    refusals = [ToolMessage(f"Not run. The reviewer rejected it: {decision}", tool_call_id=call["id"])
                for call in calls]
    return Command(goto="model", update={"messages": refusals})


def route(state):
    calls = state["messages"][-1].tool_calls
    if not calls:
        return END
    if any(call["name"] == "file_refund_request" for call in calls):
        return "review"
    return "tools"

Part B's graph. `build_desk` holds the wiring, because Task 2 builds the graph a second time with another route.

In [6]:
def build_desk(route):
    builder = StateGraph(DeskState)

    builder.add_node("model", model_node)
    builder.add_node("tools", tools_node)
    builder.add_node("review", review, destinations=("tools", "model"))

    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", route, ["tools", "review", END])
    builder.add_edge("tools", "model")

    return builder.compile(checkpointer=InMemorySaver())


desk = build_desk(route)

print(list(desk.get_graph().nodes))

['__start__', 'model', 'tools', 'review', '__end__']


## Task 1: Reject a refund with a reason

The request names invoice 229 and 13.86 dollars. It goes on its own thread id.

In [7]:
REQUEST_229 = ("Puja Srivastava was charged twice for invoice 229 (13.86 US dollars). "
               "Please file a refund request for the duplicate charge.")

config = {"configurable": {"thread_id": "refund-puja-229"}}

paused = desk.invoke({"messages": [HumanMessage(REQUEST_229)]}, config)

print("waiting on:", paused["__interrupt__"][0].value)
show_requests()

waiting on: [{'tool': 'file_refund_request', 'args': {'invoice_id': 229, 'amount': 13.86, 'reason': 'Duplicate charge'}}]
refund_requests: empty


The run stopped at `review` with one waiting call, for invoice 229 and 13.86 dollars, and the table is empty.

The reviewer rejects. In our `review` node any resume value other than `approve` is a rejection, and its text is
the reason.

In [8]:
REASON = "billing shows one charge for invoice 229, not two. Do not file it again; tell the rep."

done = desk.invoke(Command(resume=REASON), config)

show(done["messages"][-2:])
print("next node:", desk.get_state(config).next)
show_requests()

tool   Not run. The reviewer rejected it: billing shows one charge for invoice 229, not two. Do not file it again; tell the rep.
ai     The refund request for invoice 229 was rejected: billing shows only one charge, not two. I did not file another request.
next node: ()
refund_requests: empty


The `tool` message is the refusal that `review` wrote, with our reason in it. The model's reply tells the rep that the request for invoice 229 was rejected, because billing shows one charge and not two, and that nothing more was filed. `next` is empty, and so is the table.

What `review` did with the rejection:

1. `interrupt(...)` returned the reason.
2. The node did not send the run to `tools`, so `file_refund_request` never ran. The table is still empty.
3. It answered the model's tool call with a `ToolMessage` that starts *Not run. The reviewer rejected it:* and
   carries the reason, and sent the run back to `model`.
4. The model read that message and wrote the reply for the rep.

## Task 2: Let small refunds through

The rule is one more condition in `route`: a `file_refund_request` call goes to `review` only when its `amount` is
5 dollars or more. A smaller one goes to `tools` like a lookup.

In [9]:
REVIEW_FROM = 5.00


def route_by_amount(state):
    calls = state["messages"][-1].tool_calls
    if not calls:
        return END
    if any(call["name"] == "file_refund_request" and call["args"]["amount"] >= REVIEW_FROM for call in calls):
        return "review"
    return "tools"


desk = build_desk(route_by_amount)

Helena's request is for 1.98 dollars.

In [10]:
HELENA_REQUEST = ("Helena Holý was charged twice for invoice 393 (1.98 US dollars). "
                  "Please file a refund request for the duplicate charge.")

small = desk.invoke({"messages": [HumanMessage(HELENA_REQUEST)]}, {"configurable": {"thread_id": "refund-helena-393"}})

print("stopped for review:", "__interrupt__" in small)
show(small["messages"][-2:])
show_requests()

stopped for review: False
tool   refund request 1 filed: invoice 393, 1.98 US dollars
ai     I filed a refund request for $1.98 on invoice 393 due to the duplicate charge. The request will be reviewed before any money is refunded.
refund_requests: [(1, 393, 1.98, 'Duplicate charge; Helena Holý was charged twice for this invoice.')]


`stopped for review: False`. The call for 1.98 dollars went from `model` straight to `tools`. The request was filed at once, and the table has a row for invoice 393.

Puja's request for invoice 284 is for 8.91 dollars.

In [11]:
PUJA_REQUEST = ("Puja Srivastava was charged twice for invoice 284 (8.91 US dollars). "
                "Please file a refund request for the duplicate charge.")

large = desk.invoke({"messages": [HumanMessage(PUJA_REQUEST)]}, {"configurable": {"thread_id": "refund-puja-284"}})

print("stopped for review:", "__interrupt__" in large)
print("waiting on:", large["__interrupt__"][0].value)
show_requests()

stopped for review: True
waiting on: [{'tool': 'file_refund_request', 'args': {'invoice_id': 284, 'amount': 8.91, 'reason': 'Duplicate charge reported by Puja Srivastava'}}]
refund_requests: [(1, 393, 1.98, 'Duplicate charge; Helena Holý was charged twice for this invoice.')]


`stopped for review: True`. The call for 8.91 dollars waits at `review`, and the table has no row for invoice 284.

The amount in the call is a number the model wrote from the rep's message. `route_by_amount` compares that number
with 5.00 in plain Python, the same way every time. Which refunds a person sees is decided by an `if` in our code,
and never by the model.